## 引导 + 配置

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "llm_lora" else Path.cwd()
sys.path.insert(0, str(ROOT))

from common.config import load_config, with_overrides
from common.utils import set_seed, get_device, ensure_dir

cfg = load_config("config.yaml")
cfg = with_overrides(cfg, epochs=3)
cfg

## 数据

In [ ]:
from datasets import Dataset

def make_toy_dataset(cfg):
    texts = [
        "### Instruction:\n1+1=?### Response:\n2",
        "### Instruction:\nWhat is AI?\n### Response:\nArtificial Intelligence"
    ] * 10
    return Dataset.from_dict({"text": texts})

## 分词

In [ ]:
def tokenize_dataset(ds, tokenizer, max_length):
    def _fn(ex):
        out = tokenizer(ex["text"], truncation=True, max_length=int(max_length), padding="max_length")
        out["labels"] = out["input_ids"].copy()
        return out
    return ds.map(_fn, remove_columns=["text"])

## 模型

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

def build_tokenizer(cfg):
    tok = AutoTokenizer.from_pretrained(cfg["model_name"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def build_lora_model(cfg):
    model = AutoModelForCausalLM.from_pretrained(cfg["model_name"])
    lora_cfg = LoraConfig(
        r=int(cfg["lora_r"]), lora_alpha=int(cfg["lora_alpha"]),
        lora_dropout=float(cfg["lora_dropout"]),
        target_modules=["c_attn"], # tiny-gpt2 用 c_attn，其他模型按需改
        task_type="CAUSAL_LM"
    )
    return get_peft_model(model, lora_cfg)

## 训练

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from common.checkpoint import save_adapter

def run_training(cfg, out_dir):
    set_seed(int(cfg["seed"]))
    tok = build_tokenizer(cfg)
    ds = tokenize_dataset(make_toy_dataset(cfg), tok, cfg["max_length"])
    model = build_lora_model(cfg)

    args = TrainingArguments(
        output_dir=str(out_dir),
        per_device_train_batch_size=int(cfg["batch_size"]),
        num_train_epochs=float(cfg["epochs"]),
        learning_rate=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
        logging_steps=1,
        save_strategy="no",
        report_to=[],
        use_cpu=(get_device(cfg["device"]).type == "cpu")
    )
    collator = DataCollatorForLanguageModeling(tok, mlm=False)
    trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
    trainer.train()
    save_adapter(model, out_dir / "adapter")
    return trainer, model, tok

out_dir = ensure_dir(cfg["output_dir"])
trainer, model, tok = run_training(cfg, out_dir)

## 推理

In [ ]:
def generate(model, tok, prompt, max_new_tokens=32):
    model.eval()
    inputs = tok(prompt, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

print(generate(model, tok, "### Instruction:\n1+1=?### Response:\n"))

## loss 曲线

In [ ]:
from common.plot import plot_loss_curve
from common.utils import ensure_dir

out_dir = ensure_dir(cfg["output_dir"])

logs = [x for x in trainer.state.log_history if "loss" in x]
steps = [x["step"] for x in logs]
losses = [x["loss"] for x in logs]

plot_loss_curve(steps, losses, out_dir / "loss.png")